# Illustrative Example: Synthetic Spatial PCA Demo

This notebook recreates the lightweight illustrative example from the paper using synthetic input data tracked in this repository.

It uses the repository functions directly:

- `build_univariate_window_matrix`
- `fit_spca`
- `rank_spca_windows`
- `plot_top_windows_overlay`

Input data lives in `data/Illustrative Example Input Data/`. Generated figures and tables are written to `outputs/Illustrative_Example/`, which is ignored by Git.

In [ ]:
from pathlib import Path
import json
import os

repo_root = Path.cwd().resolve()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent
os.environ.setdefault("MPLCONFIGDIR", str(repo_root / ".matplotlib-cache"))

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display
from rasterio.transform import Affine
from shapely.geometry import box

import sys
sys.path.insert(0, str(repo_root / "src"))

from spatial_pca.spca.windows import build_univariate_window_matrix
from spatial_pca.spca.pca import fit_spca
from spatial_pca.spca.ranking import rank_spca_windows
from spatial_pca.validation.footprint_recovery import plot_top_windows_overlay

plt.rcParams["figure.dpi"] = 120


In [ ]:
data_dir = repo_root / "data" / "Illustrative Example Input Data"
output_dir = repo_root / "outputs" / "Illustrative_Example"
output_dir.mkdir(parents=True, exist_ok=True)

field = np.loadtxt(data_dir / "synthetic_field.csv", delimiter=",", skiprows=1)
metadata = json.loads((data_dir / "metadata.json").read_text())
known_windows = pd.read_csv(data_dir / "known_deposit_windows.csv")

variable_name = metadata["variable_name"]
win_h = int(metadata["window_height"])
win_w = int(metadata["window_width"])
stride_y = int(metadata["stride_y"])
stride_x = int(metadata["stride_x"])
training_window_index = int(metadata["training_window_index"])

known_windows


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4.5))
im = ax.imshow(field, cmap="viridis", origin="upper")
ax.set_title("Illustrative Example Input Data")
ax.set_xlabel("Column")
ax.set_ylabel("Row")
fig.colorbar(im, ax=ax, fraction=0.045, pad=0.04, label=variable_name)
fig.tight_layout()
fig.savefig(output_dir / "illustrative_input_field.png", dpi=200)
plt.show()


In [ ]:
n_rows = (field.shape[0] - win_h) // stride_y + 1
n_cols = (field.shape[1] - win_w) // stride_x + 1

def window_row_col(window_index: int) -> tuple[int, int]:
    row = int(window_index) // n_cols
    col = int(window_index) % n_cols
    return row, col

train_row, train_col = window_row_col(training_window_index)
training_template = field[train_row:train_row + win_h, train_col:train_col + win_w]

window_matrix = build_univariate_window_matrix(
    raster=field,
    deposit_template=training_template,
    variable_name=variable_name,
    stride_y=stride_y,
    stride_x=stride_x,
)

pca_result = fit_spca(
    window_matrix.data_for_pca,
    var_name=variable_name,
    patch_size=window_matrix.window_shape,
)

ranking = rank_spca_windows(
    scores=pca_result.scores,
    eigvals=pca_result.eigvals,
    deposit_index=window_matrix.deposit_index,
    k_pcs=None,
)

n_windows = window_matrix.combined_sliding_windows.shape[0]
keep = [
    int(idx) < n_windows and int(idx) != training_window_index
    for idx in ranking.ranked_idx
]
ranked_prediction_indices = ranking.ranked_idx[keep]
ranked_prediction_distances = ranking.ranked_dists[keep]

top_n = 10
top_table = pd.DataFrame({
    "rank": np.arange(1, top_n + 1),
    "window_index": ranked_prediction_indices[:top_n],
    "distance": ranked_prediction_distances[:top_n],
})
top_table.to_csv(output_dir / "illustrative_top_10_predicted_windows.csv", index=False)
top_table


In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.2))
weights = ranking.weights
ax.bar(np.arange(1, len(weights) + 1), weights, color="#4477aa")
ax.set_title("Deposit-specific PC Weights")
ax.set_xlabel("Principal component")
ax.set_ylabel("Weight")
ax.set_ylim(0, max(weights.max() * 1.15, 0.05))
fig.tight_layout()
fig.savefig(output_dir / "illustrative_pc_weights.png", dpi=200)
plt.show()


In [ ]:
N = field.shape[0]
extent = (0, N, 0, N)
transform = Affine.translation(0, N) * Affine.scale(1, -1)

def window_geometry(window_index: int):
    row, col = window_row_col(window_index)
    x0 = col
    x1 = col + win_w
    y1 = N - row
    y0 = y1 - win_h
    return box(x0, y0, x1, y1)

top_windows_gdf = gpd.GeoDataFrame(
    {
        "rank": np.arange(1, top_n + 1),
        "score": ranked_prediction_distances[:top_n],
        "window_id": ranked_prediction_indices[:top_n],
    },
    geometry=[window_geometry(idx) for idx in ranked_prediction_indices[:top_n]],
    crs="EPSG:3857",
)

deposit_window_ids = [training_window_index] + known_windows["window_index"].astype(int).tolist()
deposit_names = ["Training Pattern"] + known_windows["label"].astype(str).tolist()
deposits_gdf = gpd.GeoDataFrame(
    {"name": deposit_names, "window_index": deposit_window_ids},
    geometry=[window_geometry(idx) for idx in deposit_window_ids],
    crs="EPSG:3857",
)

plot_path = plot_top_windows_overlay(
    top_windows_gdf=top_windows_gdf,
    deposits_gdf=deposits_gdf,
    reference_deposit_index=0,
    background_layers={
        variable_name: {
            "array": field,
            "extent": extent,
            "vmin": float(np.nanmin(field)),
            "vmax": float(np.nanmax(field)),
        }
    },
    transform=transform,
    output_path=output_dir / f"{variable_name}_Top_{top_n}_Predicted_Windows.png",
    title=f"{variable_name}: Top {top_n} Prediction Windows",
    image_cmap="viridis",
)

display(Image(filename=str(plot_path)))


In [ ]:
known_geometries = [window_geometry(idx) for idx in known_windows["window_index"].astype(int)]
min_cover = 0.5
hits = []
recovered = set()

for rank, geom in enumerate(top_windows_gdf.geometry, start=1):
    for known_idx, known_geom in enumerate(known_geometries):
        cover = geom.intersection(known_geom).area / known_geom.area
        if cover >= min_cover:
            recovered.add(known_idx)
    hits.append(len(recovered))

hit_table = pd.DataFrame({
    "rank": np.arange(1, len(hits) + 1),
    "known_deposits_recovered": hits,
    "recovery_fraction": np.asarray(hits) / len(known_geometries),
})
hit_table.to_csv(output_dir / "illustrative_known_deposit_recovery.csv", index=False)

fig, ax = plt.subplots(figsize=(5.5, 3.5))
ax.plot(hit_table["rank"], hit_table["recovery_fraction"], marker="o", color="#228833")
ax.set_title("Known Deposit Recovery")
ax.set_xlabel("Top N predicted windows")
ax.set_ylabel("Fraction recovered")
ax.set_ylim(-0.03, 1.03)
ax.grid(True, alpha=0.25)
fig.tight_layout()
fig.savefig(output_dir / "illustrative_known_deposit_recovery.png", dpi=200)
plt.show()

hit_table


## Outputs

After running the notebook, check `outputs/Illustrative_Example/` for:

- `Synthetic_GP_Top_10_Predicted_Windows.png`
- `illustrative_top_10_predicted_windows.csv`
- `illustrative_known_deposit_recovery.csv`
- simple diagnostic PNGs for the input field, PC weights, and recovery curve